# RSMI-NE — q=3 Potts model

Train, visualize, and estimate the scaling dimension of the leading primary
operators of the 2D 3-state Potts CFT:

- the **energy** operator ε (Z₃-trivial sector), and
- the **spin** operator σ, realized through its two real components
  σ_cos and σ_sin (the two complex-conjugate Z₃ characters).

The operators differ only in the Z₃ character used for the symmetry
projection; data, V/E geometry, network hyperparameters, and analysis are
shared. The flow is:

1. load MC samples and build the V/E geometry,
2. set the shared hyperparameters,
3. train each sector with `train_operator`,
4. visualize each with `visualize_operator`,
5. estimate dimensions (neural vs. naive) with `measure_dimensions`.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   # or "3" for only errors

import sys
import re
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from enum import IntEnum

# Run from the repo root so that the Data/ directory and the RSMI package both resolve
sys.path.insert(0, os.path.abspath(os.getcwd()))
from RSMI import (
    CoarseGrainer,
    SeparableCritic,
    train_RSMI_optimiser,
    augment_by_permutations,
    multivariate_fit,
    plot_circle_graph,
    DrawCircle,
    build_tfrecord_dataset,
    samples_to_ve_dataset,
    run_connected_correlation,
    scaling_dimensions_from_correlations,
    nearest_neighbors_hamming_one,
)

## Symmetry projection

`SymmetrizeCG` averages the coarse-grainer over the D₄ lattice point-group
permutations and the three global Z₃ Potts shifts (via `Mod3Layer`), weighted
by a one-dimensional Z₃ character. `Z3Sector.TRIVIAL` selects the Z₃-invariant
(energy) sector; `Z3Sector.E_COS` and `Z3Sector.E_SIN` are the real cosine/sine
combinations of the two conjugate characters that together span the spin
(σ) sector.

In [ ]:
class Z3Sector(IntEnum):
    NONE = 0
    TRIVIAL = 1
    E_COS = 2
    E_SIN = 3


class Mod3Layer(tf.keras.layers.Layer):
    """
    A Keras layer that adds n to the input (element-wise) and then takes modulo 3.
    """
    def __init__(self, n, **kwargs):
        super(Mod3Layer, self).__init__(**kwargs)
        self.n = n

    def call(self, inputs, training=None):
        return tf.math.floormod(inputs + self.n, 3)

def SymmetrizeCG(F, permutations, z3_sector, V_size):
    """Average F over the D4 spatial permutations and the three global Z3 Potts
    shifts (via Mod3Layer), weighted by the chosen one-dimensional Z3 character:
    TRIVIAL -> energy sector; E_COS / E_SIN -> the real cos/sin components of the
    spin sector. Returns a Keras model mapping V to the symmetry-projected scalar."""
    input_layer = tf.keras.Input(shape=(V_size,))  # Exclude batch size
    permutated_input = tf.keras.layers.Lambda(lambda x: tf.gather(x, indices=permutations, axis=-1))(input_layer)  # spatial symmetrization; axis (-1) is the data dimension
    layer1 = Mod3Layer(n=1)
    layer2 = Mod3Layer(n=2)
    permutated_F0 = tf.math.reduce_mean(F(permutated_input), axis=-2)  # axis (-2) is the permutations dimension
    permutated_F1 = tf.math.reduce_mean(F(layer1(permutated_input)), axis=-2)
    permutated_F2 = tf.math.reduce_mean(F(layer2(permutated_input)), axis=-2)
    if z3_sector == Z3Sector.NONE:
        return tf.keras.Model(inputs=input_layer, outputs=(permutated_F0))
    elif z3_sector == Z3Sector.TRIVIAL:
        return tf.keras.Model(inputs=input_layer, outputs=(1/3*(permutated_F0 + permutated_F1 + permutated_F2)))
    elif z3_sector == Z3Sector.E_COS:
        return tf.keras.Model(inputs=input_layer, outputs=(1/3*(permutated_F0 - 0.5*permutated_F1 - 0.5*permutated_F2)))  # cos(2pi/3)
    elif z3_sector == Z3Sector.E_SIN:
        return tf.keras.Model(inputs=input_layer, outputs=(1/np.sqrt(2)*(permutated_F1 - permutated_F2)))  # sin(2pi/3)
    else:
        assert(False)

## Load data and define the V/E geometry

A single linear size `L` is used to *fit* the operators (the dimension
measurement later scans many sizes). The visible region `V` is a circular disk
of radius `size`; the environment `E` is a circular shell offset by `buf`.

In [ ]:
# Training data: single linear size L used to fit the operators
train_dataset_dir = os.path.join("Data", "potts", "L40")

batch_size = 16000

train_dataset, test_dataset, L, dim, train_size, test_size = build_tfrecord_dataset(
    train_dataset_dir,
    batch_size=batch_size,
    train_frac=0.9,
)
print("L=%d, dim=%d, train samples=%d, test samples=%d" % (L, dim, train_size, test_size))

buf = 4
shape = "sphere"
size = 3
print("L, buf, shape, size:", L, buf, shape, size)

V_indices, E_indices, V_size, E_size, permutations, samples_to_ve = samples_to_ve_dataset(
    L, shape=shape, buf=buf, size=size, use_symmetry=True, dim=dim,
)
print("V_size, E_size:", V_size, E_size, "permutations shape:", permutations.shape)

train_dataset = samples_to_ve(train_dataset)
test_dataset = samples_to_ve(test_dataset)

## Training hyperparameters

Shared across all sectors — only the symmetry projection differs between the
energy and spin operators.

In [ ]:
# Training params: critic, opt, CG (shared across all operators)
cur_critic = SeparableCritic
critic_params = {
    "layers_V": 3,
    "layers_E": 3,
    "embed_dim": 256,
    "hidden_dim_V": 64,
    "hidden_dim_E": 2 * E_size,
    "activation": "relu",
}
c = cur_critic(**critic_params)
opt_params = {
    "patience": 1,
    "batch_size": batch_size,
    "iterations": 1,
    "learning_rate": 3e-4,
    "N_samples": train_size,
    "do_noise": True,
    "noise": 0.1,
}
cycles_per_iteration = int(np.ceil(opt_params["N_samples"] / batch_size))
CG_params = {
    "init_temperature": 1.0,
    "min_temperature": 0.2,
    "layers": 3,
    "layer_width": 2 * V_size,
    "activation": "relu",
    "batchNorm_scale": 0.1,
    "kernel_regularization_weight": 0,
    "bias_regularization_weight": 0,
    "activity_regularization_weight": 0,
    "learn_beta": True,
    "dataset_path": train_dataset_dir,
    "cur_critic": c.critic_name,
}
additional_data = {
    "V_indices": V_indices
}
full_relax_factor = 0.95
CG_params["relaxation_rate"] = np.log(
    CG_params["init_temperature"] / CG_params["min_temperature"]
) / (cycles_per_iteration * opt_params["iterations"] * full_relax_factor)
bound = "infonce" 

## Training driver

`train_operator` runs RSMI-NE for one Z₃ sector and returns the trained
`CoarseGrainer` together with the log directory its encoder was saved to. The
projected, deterministic operator is available as `CG.encoder`.

In [ ]:
def train_operator(z3_sector, cg_name):
    """Train one RSMI-NE operator in a given Z3 sector.

    Returns (CG, log_dir). The symmetry-projected operator is CG.encoder."""
    params = dict(CG_params)
    params["cg_name"] = cg_name
    symmetrization_operation = lambda model: SymmetrizeCG(model, permutations, z3_sector, V_size)
    return train_RSMI_optimiser(
        CoarseGrainer,
        params,
        critic_params,
        opt_params,
        additional_data,
        train_dataset,
        "logs",
        cur_critic,
        symmetrization_operation,
        bound=bound,
        test_dataset=test_dataset,
        dataset_dir=train_dataset_dir,
    )

## Train the energy operator (ε, Z₃-trivial)

In [ ]:
CG_eps, log_dir_eps = train_operator(Z3Sector.TRIVIAL, "Rotations and Mirror (Z3 trivial)")
print("Energy operator saved to logs/%s" % log_dir_eps)

## Train the spin operator, cosine component (σ_cos, Z₃ cos)

In [ ]:
CG_sigma_cos, log_dir_sigma_cos = train_operator(Z3Sector.E_COS, "Rotations and Mirror (Z3 cos)")
print("Spin (cos) operator saved to logs/%s" % log_dir_sigma_cos)

## Train the spin operator, sine component (σ_sin, Z₃ sin)

In [ ]:
CG_sigma_sin, log_dir_sigma_sin = train_operator(Z3Sector.E_SIN, "Rotations and Mirror (Z3 sin)")
print("Spin (sin) operator saved to logs/%s" % log_dir_sigma_sin)

## Visualize the learned operators

Fit an orbit-symmetric polynomial surrogate to each operator and draw it on the
circular support. With `degree=1`, `DrawCircle` shows the single-site weights;
set `degree=2` to inspect pairwise couplings via `plot_circle_graph`. (2D only.)

In [ ]:
def visualize_operator(CG, title, degree):
    """Fit a symmetric polynomial surrogate to CG.encoder and plot it on the circle layout."""
    print("--- %s ---" % title)
    V, _ = next(iter(test_dataset))
    V = augment_by_permutations(V, permutations)
    model, poly = multivariate_fit(V.numpy(), CG.encoder(V).numpy(), degree, interaction_only=True)
    if dim != 2:
        print("Skipping 2D circle plot for dim=%d" % dim)
        return model, poly
    if degree == 1:
        DrawCircle(model.coef_.ravel(), V_indices, node_size=100)
        return model, poly
    elif degree == 2:
        coef = model.coef_.ravel()
        s = np.argsort(np.abs(coef))[::-1]
        feat_names = np.asarray(poly.get_feature_names_out())
        out = list(zip(feat_names[s], np.round(coef[s], 4)))
        fig, ax = plot_circle_graph(out, V_indices, node_size=96, top_k=len(feat_names), min_abs=None, fc="white", ec="black")
        plt.show()
    return model, poly

### Energy operator

In [ ]:
model_eps, poly_eps = visualize_operator(CG_eps, "Energy operator (epsilon, Z3-trivial)", degree=2)

### Spin operator (cosine component)

In [ ]:
model_scos, poly_scos = visualize_operator(CG_sigma_cos, "Spin operator (sigma cos, Z3)", degree=1)

### Spin operator (sine component)

In [ ]:
model_ssin, poly_ssin = visualize_operator(CG_sigma_sin, "Spin operator (sigma sin, Z3)", degree=1)

## Naive lattice operators

Conventional estimators used as the baseline for the dimension comparison, built
from the complex Potts phase exp(2πi s_j / 3): the real/imaginary parts of the
magnetization (1/N) Σ_j exp(2πi s_j / 3) for the spin (σ_cos / σ_sin), and the
real part of the nearest-neighbor bond product Σ_⟨ij⟩ exp(2πi(s_i − s_j)/3) for
the energy (ε).

In [ ]:
V_indices_list = [tuple(int(x) for x in p) for p in V_indices]
bulk_energy_indices = nearest_neighbors_hamming_one(V_indices_list)
b1 = [i[0] for i in bulk_energy_indices]
b2 = [i[1] for i in bulk_energy_indices]

# Complex Potts magnetization phase per site: exp(2*pi*i*s/3). V has shape (batch*L*L, V_size).

@tf.function
def SigmaCos(V):
    """Naive spin operator: real part of the Potts magnetization (1/N) sum_j exp(2*pi*i*s_j/3)."""
    return tf.math.reduce_mean(tf.math.real(tf.exp(2j * np.pi * tf.cast(V, tf.complex64) / 3)), axis=1)

@tf.function
def SigmaSin(V):
    """Naive spin operator: imaginary part of the Potts magnetization (1/N) sum_j exp(2*pi*i*s_j/3)."""
    return tf.math.reduce_mean(tf.math.imag(tf.exp(2j * np.pi * tf.cast(V, tf.complex64) / 3)), axis=1)

@tf.function
def Epsilon(V):
    """Naive energy operator: real part of the nearest-neighbor bond product over V."""
    sigma = tf.exp(2j * np.pi * tf.cast(V, tf.complex64) / 3)
    return tf.math.reduce_mean(
        tf.math.real(tf.gather(sigma, b1, axis=1) * tf.math.conj(tf.gather(sigma, b2, axis=1))), axis=1
    )

## Measure scaling dimensions

Uses Sandvik's finite-size scaling method. This requires samples at several
linear sizes under `Data/potts/L{L}/` (see the README for how to generate
them).

In [ ]:
scan_dataset_dir = os.path.join("Data", "potts")  # parent dir holding L{L} subdirs at many sizes

def discover_L_dirs(dataset_dir):
    subdirs = [
        d for d in os.listdir(dataset_dir)
        if os.path.isdir(os.path.join(dataset_dir, d)) and re.match(r"^L(\d+)$", d)
    ]
    subdirs.sort(key=lambda d: int(re.match(r"^L(\d+)$", d).group(1)))
    return subdirs

def measure_dimensions(operator, V_indices, *, batch_size=50, sample_size=256,
                       time_threshold=60, log_err_threshold=1e-1, seed=42):
    """Estimate scaling dimensions of `operator` across all available L

    Returns (corr_dict, dimensions)."""
    corr_dict = {}
    for subdir in discover_L_dirs(scan_dataset_dir):
        Lc = int(re.match(r"^L(\d+)$", subdir).group(1))
        data_dir = os.path.join(scan_dataset_dir, subdir)
        print(f"L = {Lc}: estimating connected correlation ...")
        (corr_tuple, result) = run_connected_correlation(
            operator, data_dir, V_indices,
            batch_size=batch_size, sample_size=sample_size, max_batches=None,
            time_threshold=time_threshold, log_err_threshold=log_err_threshold, seed=seed,
        )
        corr_dict[Lc] = corr_tuple
        print(
            f"  L = {Lc:<3d} | connected correlation = {result['val']:.6e} "
            f"+/- {result['err']:.2e} (relative log-error {result['log_err']:.4f})"
        )
    return corr_dict, scaling_dimensions_from_correlations(corr_dict)

### Energy: neural vs. naive

In [ ]:
corr_eps_neural, dim_eps_neural = measure_dimensions(CG_eps.encoder, V_indices, batch_size=50, log_err_threshold=1e-2)
corr_eps_naive,  dim_eps_naive  = measure_dimensions(Epsilon,        V_indices, batch_size=50, log_err_threshold=1e-2)
print("epsilon neural:", dim_eps_neural)
print("epsilon naive :", dim_eps_naive)

### Spin (cosine): neural vs. naive

In [ ]:
corr_scos_neural, dim_scos_neural = measure_dimensions(CG_sigma_cos.encoder, V_indices, batch_size=50, log_err_threshold=1e-4)
corr_scos_naive,  dim_scos_naive  = measure_dimensions(SigmaCos,             V_indices, batch_size=50, log_err_threshold=1e-4)
print("sigma(cos) neural:", dim_scos_neural)
print("sigma(cos) naive :", dim_scos_naive)

### Spin (sine): neural vs. naive

In [ ]:
corr_ssin_neural, dim_ssin_neural = measure_dimensions(CG_sigma_sin.encoder, V_indices, batch_size=50, log_err_threshold=1e-4)
corr_ssin_naive,  dim_ssin_naive  = measure_dimensions(SigmaSin,             V_indices, batch_size=50, log_err_threshold=1e-4)
print("sigma(sin) neural:", dim_ssin_neural)
print("sigma(sin) naive :", dim_ssin_naive)